In [ ]:
import pandas as pd
import numpy as np

train = pd.read_csv("train (1).csv")
test = pd.read_csv("test (2).csv")

print(train.shape)
print(test.shape)

train.head()


## Checking the data

First I checked the columns and missing values. Some columns are text and some values like price and mileage are not in normal number format.


In [ ]:
print(train.info())
print(train.isnull().sum())


In [ ]:
# price has $ and commas, so I removed them

train["price"] = train["price"].str.replace("$", "", regex=False)
train["price"] = train["price"].str.replace(",", "", regex=False)
train["price"] = train["price"].astype(float)

# mileage is also written like "12345 mi."
train["milage"] = train["milage"].str.replace(",", "", regex=False)
train["milage"] = train["milage"].str.replace(" mi.", "", regex=False)
train["milage"] = train["milage"].astype(float)

test["milage"] = test["milage"].str.replace(",", "", regex=False)
test["milage"] = test["milage"].str.replace(" mi.", "", regex=False)
test["milage"] = test["milage"].astype(float)

print(train[["price", "milage"]].head())


## Removing a few columns

I removed `id` because it is just an identifier. I also removed `clean_title` because it does not give much useful information in this dataset.


In [ ]:
test_ids = test["id"]

train = train.drop(columns=["id", "clean_title"])
test = test.drop(columns=["id", "clean_title"])

X = train.drop(columns=["price"])
y = train["price"]

print(X.shape)


## Making the model

There are many text columns such as brand, model and fuel type. Random Forest cannot directly use these, so I am converting the text columns using one-hot encoding.

Missing numerical values are filled with the median.


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error

cat_cols = X.select_dtypes(include=["object"]).columns
num_cols = X.select_dtypes(exclude=["object"]).columns

num_process = Pipeline([
    ("fill", SimpleImputer(strategy="median"))
])

cat_process = Pipeline([
    ("fill", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocess = ColumnTransformer([
    ("numbers", num_process, num_cols),
    ("categories", cat_process, cat_cols)
])

model = RandomForestRegressor(
    n_estimators=250,
    random_state=42,
    n_jobs=-1,
    max_features="sqrt"
)

pipe = Pipeline([
    ("preprocess", preprocess),
    ("model", model)
])


## Testing the model

I am keeping 20% of the data for checking how well the model works before using it on the test data.


In [ ]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42
)

pipe.fit(X_train, y_train)

pred_valid = pipe.predict(X_valid)

rmse = mean_squared_error(y_valid, pred_valid) ** 0.5
mae = mean_absolute_error(y_valid, pred_valid)

print("RMSE:", round(rmse, 2))
print("MAE:", round(mae, 2))


The score is not perfect, but it gives an idea of how the model is performing. There are some expensive cars in the data, so a few wrong predictions can make RMSE quite large.


In [ ]:
# Now training the model on all the training data

pipe.fit(X, y)

test_pred = pipe.predict(test)

submission = pd.DataFrame({
    "id": test_ids,
    "price": test_pred
})

submission.head()


In [ ]:
# saving the final file

submission.to_csv("sample_submission.csv", index=False)

print("File saved successfully")
print(submission.shape)
